# 01 — Data assembly

Builds one clean feature table for the Big Island coffee analysis, from inputs that have all been corrected or re-measured. Replaces the scattered `data_wrangling/*.py` steps, which remain in the repo as artefacts.

**What changed from the old pipeline, and why it matters**

| input | old | now |
|---|---|---|
| annual rainfall | ~4 km product, Kona 6,471 mm / Ka'u 741 mm | 250 m Rainfall Atlas, Kona 1,321 / Ka'u 1,783 |
| seasonal rainfall | `DRY_MONTHS = [6,7,8,9]` for both districts | window-free, per-cell, from 384 monthly rasters |
| elevation | `elev_mean` **and** `elev_dev_mean` (r = 1.000) | `elev_mean` only |

The old rainfall column had the two districts **inverted** — it made Kona nine times wetter than Ka'u when Ka'u is in fact the wetter of the two — and the old dry-season window turned out to be Kona's four *wettest* months. Every "Kona wet / Ka'u dry" statement in the previous pipeline descends from those two errors.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

DATA = '../data'
OUT  = 'data'
os.makedirs(OUT, exist_ok=True)
MONTHS = [f'month_{i:02d}' for i in range(1, 13)]

## Terrain and soil

Straight from the assembled grid. `elev_dev_mean` is deliberately excluded: it is `elev_mean` minus a scalar, correlating at r = 1.000, so it double-weights elevation in any distance metric and splits elevation's importance in any feature ranking.

In [2]:
base = pd.read_pickle(f'{DATA}/plot_all_features.pkl')

TERRAIN = ['elev_mean', 'slope_max', 'aspect_sin', 'aspect_cos',
           'total_relief', 'local_relief', 'dist_coast_m']
SOIL    = ['drain_ord', 'restrictiondepth_cm', 'awc_mean', 'ph_0_30cm',
           'om_0_30cm', 'sand_0_30cm', 'silt_0_30cm', 'clay_0_30cm', 'cec_0_30cm']
assert 'elev_dev_mean' not in TERRAIN

df = base[['plot_id', 'region', 'label'] + TERRAIN + SOIL].copy()
print(f'grid cells {len(df):,}, labelled coffee {int(df.label.sum())}')

grid cells 10,211, labelled coffee 471


## Rainfall

Two products, deliberately. The **annual** figure comes from the Rainfall Atlas of Hawai'i (Giambelluca et al. 2013, 250 m, 1978–2007), sampled via the State GIS service. The **monthly** climatology comes from 384 HCDP rasters (1990–2021), which is what makes seasonality measurable.

They are independent, and they agree: Ka'u is wetter by a ratio of 1.35 (Atlas) and 1.26 (HCDP monthly sums). The old 4 km column said 0.11.

In [3]:
rain = pd.read_csv(f'{DATA}/rainfall_atlas_250m.csv').drop_duplicates('plot_id')
df = df.merge(rain.rename(columns={'rain_mm': 'precip_annual'}), on='plot_id', how='left')

R = pd.read_csv(f'{DATA}/rainfall_monthly_climatology.csv').drop_duplicates('plot_id').set_index('plot_id')[MONTHS]
T = pd.read_csv(f'{DATA}/temp_monthly_climatology.csv').drop_duplicates('plot_id').set_index('plot_id')[MONTHS]
assert (R.index == T.index).all(), 'monthly climatologies must be row-aligned'
print(f'monthly climatologies: {len(R)} cells x 12 months, rainfall and temperature')

monthly climatologies: 471 cells x 12 months, rainfall and temperature


## Seasonality — window-free

No fixed dry-season months. Each cell's descriptors come from its own annual cycle:

- `precip_driest_q` / `precip_wettest_q` — mm in that cell's driest / wettest three consecutive months (BIO17 / BIO16 style)
- `precip_seasonality` — coefficient of variation of monthly means (BIO15 style)
- `precip_peak_sin` / `_cos` — **phase**, as the circular mean of the cycle

Phase is the one that matters. Amplitude is near-identical between districts; the entire difference is timing. The districts' monthly cycles are anti-correlated at r = −0.58 — Kona peaks in September, Ka'u in March.

`warm_wet_coupling` is the headline variable: the temperature at which a cell's rain actually arrives, minus its annual mean temperature. Positive means rain falls in the warm season.

In [4]:
Rv, Tv = R.values, T.values
ang = 2 * np.pi * np.arange(12) / 12.0

wrap = np.concatenate([Rv, Rv[:, :2]], axis=1)
quarters = np.stack([wrap[:, i:i+3].sum(axis=1) for i in range(12)], axis=1)

px = (Rv * np.cos(ang)).sum(1) / Rv.sum(1)
py = (Rv * np.sin(ang)).sum(1) / Rv.sum(1)

seas = pd.DataFrame({
    'precip_driest_q':    quarters.min(1),
    'precip_wettest_q':   quarters.max(1),
    'precip_seasonality': 100 * Rv.std(1, ddof=0) / Rv.mean(1),
    'precip_peak_cos':    px / np.hypot(px, py),
    'precip_peak_sin':    py / np.hypot(px, py),
    'temp_mean_annual':   Tv.mean(1),
    'temp_range_annual':  Tv.max(1) - Tv.min(1),
    'temp_rain_weighted': (Tv * Rv).sum(1) / Rv.sum(1),
}, index=R.index)
seas['warm_wet_coupling'] = seas.temp_rain_weighted - seas.temp_mean_annual

df = df.merge(seas, left_on='plot_id', right_index=True, how='left')
print(f'{len(seas.columns)} derived columns added')

9 derived columns added


## Verification

These assertions encode what the corrected data must show. If an input silently reverts, the notebook fails here rather than producing a plausible-looking wrong answer downstream.

Note the coupling assertion tests **non-overlap**, not sign. 408 of Kona's 409 cells are positive; the exception (plot 10043, 22 km inland) is likely mislabelled by `SPLIT_LON`, which assigns district on longitude alone. Kona's minimum still sits above Ka'u's maximum, so the distributions are disjoint — that is the defensible claim, and "100% positive" is not.

In [5]:
stale = [c for c in df.columns if c in ('precip_dry', 'precip_wet', 'precip_dry_frac')]
assert not stale, f'stale windowed columns present: {stale}'

farms = df[df.label == 1]
k = farms[farms.region == 'kona']
q = farms[farms.region == 'kau']

assert farms.precip_annual.notna().all(),                      'farm cells missing rainfall'
assert q.precip_annual.mean() > k.precip_annual.mean(),        "Ka'u should be the wetter district"
assert k.warm_wet_coupling.min() > q.warm_wet_coupling.max(),  'coupling should not overlap'

print('verification passed\n')
print(f"  precip_annual    kona {k.precip_annual.mean():6.0f} mm    kau {q.precip_annual.mean():6.0f} mm")
print(f"  temp range       kona {k.temp_range_annual.mean():6.2f} C     kau {q.temp_range_annual.mean():6.2f} C")
print(f"  warm/wet coupling kona {k.warm_wet_coupling.mean():+5.2f} C     kau {q.warm_wet_coupling.mean():+5.2f} C")
print(f"  coupling range   kona [{k.warm_wet_coupling.min():+.3f}, {k.warm_wet_coupling.max():+.3f}]"
      f"   kau [{q.warm_wet_coupling.min():+.3f}, {q.warm_wet_coupling.max():+.3f}]")
print(f"  separation gap   {k.warm_wet_coupling.min() - q.warm_wet_coupling.max():+.3f} C (disjoint)")

verification passed

  precip_annual    kona   1321 mm    kau   1783 mm
  temp range       kona   3.23 C     kau   3.24 C
  warm/wet coupling kona +0.30 C     kau -0.15 C
  coupling range   kona [-0.017, +0.385]   kau [-0.211, -0.074]
  separation gap   +0.058 C (disjoint)


## Write

`grid_features.pkl` is the full island (10,211 cells) for the feasible-set work; `farm_features.pkl` is the 471 labelled coffee cells for district characterisation.

In [6]:
df.to_pickle(f'{OUT}/grid_features.pkl')
farms.to_pickle(f'{OUT}/farm_features.pkl')
print(f'wrote {OUT}/grid_features.pkl ({len(df):,} cells)')
print(f'wrote {OUT}/farm_features.pkl ({len(farms)} cells)')

wrote data/grid_features.pkl (10,211 cells)
wrote data/farm_features.pkl (471 cells)
